# Experimento Federado com Comunicacao Semantica

Este notebook e o painel principal da pesquisa.

Objetivos:
- Orquestrar rodadas de Aprendizado Federado (FL) nos containers.
- Monitorar convergencia (loss e acuracia global por rodada).
- Avaliar eficiencia semantica com proxy de compressao e reducao de banda.

A estrategia: os containers gravam pesos/metricas no volume compartilhado `shared_data`, e o notebook le esses dados quase em tempo real para analise.

In [ ]:
!pip install torch torchvision

In [30]:
import torchvision.datasets as datasets

# Garanta que o caminho aponta para a pasta COMPARTILHADA do seu Docker
caminho_dataset = 'shared_data/ml-data/datasets' 

# Dicionário mapeando os nomes para as classes correspondentes no PyTorch
catalogo_datasets = {
    'mnist': datasets.MNIST,
    'fashion_mnist': datasets.FashionMNIST,
    'cifar-10': datasets.CIFAR10,
    'cifar-100': datasets.CIFAR100
}

# Lista dos datasets que você quer garantir o download (pode editar se quiser baixar só um)
datasets_para_baixar = ['mnist', 'fashion_mnist', 'cifar-10', 'cifar-100']

print("📥 Iniciando o download seguro dos datasets...\n")

for nome in datasets_para_baixar:
    if nome in catalogo_datasets:
        print(f"⏳ Baixando {nome.upper()}...")
        dataset_class = catalogo_datasets[nome]
        
        # Baixa a base de treino
        dataset_class(root=caminho_dataset, train=True, download=True)
        # Baixa a base de teste
        dataset_class(root=caminho_dataset, train=False, download=True)
        
        print(f"✅ {nome.upper()} pronto e armazenado!\n")
    else:
        print(f"❌ Erro: Dataset '{nome}' não configurado no catálogo.")

print("🚀 Todos os downloads concluídos com sucesso! O Docker já pode usar os arquivos locais.")

ModuleNotFoundError: No module named 'torchvision'

In [29]:
import json
import shutil
import subprocess
from pathlib import Path

SHARED_ROOT = Path('shared_data')

to_clear = [
    SHARED_ROOT / 'ml-data',
    SHARED_ROOT / 'fl-weights',
    SHARED_ROOT / 'resultados',
]

for path in to_clear:
    if path.exists():
        print(f'[setup] limpando: {path}')
        shutil.rmtree(path)

required_dirs = [
    SHARED_ROOT / 'ml-data' / 'datasets',
    SHARED_ROOT / 'ml-data' / 'runs',
    SHARED_ROOT / 'ml-data' / 'logs',
    SHARED_ROOT / 'fl-weights',
    SHARED_ROOT / 'resultados' / 'notebook_monitor',
]

for path in required_dirs:
    path.mkdir(parents=True, exist_ok=True)
    print(f'[setup] diretorio pronto: {path}')

history_path = SHARED_ROOT / 'resultados' / 'notebook_monitor' / 'history.json'
history_path.write_text(json.dumps({"snapshots": []}, indent=2), encoding='utf-8')

print('[setup] subindo containers...')
res = subprocess.run(
    ['docker', 'compose', 'up', '-d', '--build'],
    capture_output=True,
    text=True
)
print('[setup] return code:', res.returncode)
if res.stdout:
    print('--- STDOUT ---')
    print(res.stdout)
if res.stderr:
    print('--- STDERR ---')
    print(res.stderr)

[setup] limpando: shared_data\ml-data
[setup] limpando: shared_data\fl-weights
[setup] limpando: shared_data\resultados
[setup] diretorio pronto: shared_data\ml-data\datasets
[setup] diretorio pronto: shared_data\ml-data\runs
[setup] diretorio pronto: shared_data\ml-data\logs
[setup] diretorio pronto: shared_data\fl-weights
[setup] diretorio pronto: shared_data\resultados\notebook_monitor
[setup] subindo containers...
[setup] return code: 0
--- STDOUT ---
#1 [internal] load local bake definitions
#1 reading from stdin 2.76kB 0.0s done
#1 DONE 0.0s

#2 [fl-client-1 internal] load build definition from Dockerfile
#2 ...

#3 [ml-service internal] load build definition from Dockerfile
#3 transferring dockerfile: 537B 0.0s done
#3 DONE 0.2s

#4 [fl-server internal] load build definition from Dockerfile
#4 transferring dockerfile: 789B 0.0s done
#4 DONE 0.2s

#2 [fl-client-1 internal] load build definition from Dockerfile
#2 transferring dockerfile: 753B 0.0s done
#2 DONE 0.2s

#5 [fl-client

In [28]:
import json
import time
import requests # <-- Importação adicionada aqui!
from datetime import datetime
from pathlib import Path

# 1. Configuração do Experimento
rounds = 5
clients = 3
dataset = 'mnist'
model = 'cnn_vae'
epochs = 3

payload = {
    'dataset': dataset,
    'model': model,
    'clients': clients,
    'epochs': epochs,
    'rounds': rounds,
    'awgn': {
        'enabled': True,
        'snr_db': 15
    },
    'masking': {
        'enabled': True,
        'drop_rate': 0.1,
        'fill_value': 0.0
    }
}

# 2. Disparo do Treinamento
print('[execucao] iniciando treino federado direto pelo Python...')
try:
    start_res = requests.post(
        'http://localhost:8100/training/start', 
        json=payload
    )
    print('[execucao] start status code:', start_res.status_code)
except requests.exceptions.RequestException as e:
    print('[execucao] erro de conexao ao iniciar:', e)

# 3. Loop de monitoramento (Polling)
history_path = Path('shared_data/resultados/notebook_monitor/history.json')
snapshots = []

while True:
    try:
        status_res = requests.get('http://localhost:8100/training/status')
        status = status_res.json()
        
        history = status.get('history', [])
        
        # Pega os dados da ÚLTIMA rodada fechada no histórico
        if history:
            last_entry = history[-1]
            last_round_num = last_entry.get('round')
            last_acc = last_entry.get('accuracy')
            last_loss = last_entry.get('loss')
        else:
            last_round_num = 0
            last_acc = "---"
            last_loss = "---"

        snapshot = {
            'timestamp': datetime.now().isoformat(timespec='seconds'),
            'state': status.get('state'),
            'current_round': status.get('current_round'), # Round que está rodando AGORA
            'last_consolidated_round': last_round_num,
            'accuracy': last_acc,
            'loss': last_loss
        }
        snapshots.append(snapshot)

        history_path.write_text(
            json.dumps({'snapshots': snapshots}, indent=2),
            encoding='utf-8'
        )

        print(f"[execucao] Status={snapshot['state']} | Rodada Atual={snapshot['current_round']} | "
              f"Ultima Metrica (R{last_round_num}): Acc={last_acc} Loss={last_loss}")

        if snapshot['state'] in {'done', 'error', 'stopped'}:
            print('[execucao] treino finalizado com state =', snapshot['state'])
            break

    except (requests.exceptions.RequestException, json.JSONDecodeError) as e:
        print('[execucao] aguardando servidor responder...', e)
    
    time.sleep(3)

[execucao] iniciando treino federado direto pelo Python...
[execucao] start status code: 200
[execucao] Status=starting | Rodada Atual=0 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active | Rodada Atual=1 | Ultima Metrica (R0): Acc=--- Loss=---
[execucao] Status=round_active 

KeyboardInterrupt: 

# Execute no terminal para acompanhar o treino dos clients
docker compose logs -f fl-client-1 fl-client-2 fl-client-3

In [18]:
import subprocess

print("=== EXTRAINDO LOGS DOS CLIENTES ===")

# Nomes dos serviços definidos no seu docker-compose.yml
clientes = ['fl-client-1', 'fl-client-2', 'fl-client-3']

for cliente in clientes:
    print(f"\n{'='*20} {cliente.upper()} {'='*20}")
    
    # Executa o comando docker logs pegando apenas as últimas 15 linhas
    res = subprocess.run(
        ['docker', 'compose', 'logs', '--tail', '5', cliente], 
        capture_output=True, 
        text=True
    )
    
    if res.stdout:
        print(res.stdout.strip())
    if res.stderr:
        print(res.stderr.strip())

=== EXTRAINDO LOGS DOS CLIENTES ===

==================== FL-CLIENT-1 ====================
fl_client_1  | [03:45:09] [client-1] [round 5] Local weights saved â†’ /app/data/fl-weights/client_1_round_5.pth
fl_client_1  | [03:45:09] [client-1] [round 5] âœ“ Submission accepted by server
fl_client_1  | [03:45:09] [client-1] [round 5] Done. Waiting for next round...
fl_client_1  | [03:45:11] [client-1] Server reports state='done' â€” training complete. Loop finished.
fl_client_1  | [03:45:11] [client-1] Training loop exited.

==================== FL-CLIENT-2 ====================
fl_client_2  | [03:45:08] [client-2] [round 5] Local weights saved â†’ /app/data/fl-weights/client_2_round_5.pth
fl_client_2  | [03:45:08] [client-2] [round 5] âœ“ Submission accepted by server
fl_client_2  | [03:45:08] [client-2] [round 5] Done. Waiting for next round...
fl_client_2  | [03:45:10] [client-2] Server reports state='done' â€” training complete. Loop finished.
fl_client_2  | [03:45:10] [client-2] Traini

In [21]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# 1. Leitura do JSON limpo
history_path = Path('shared_data/resultados/notebook_monitor/history.json')
if not history_path.exists():
    raise FileNotFoundError(f'Arquivo nao encontrado: {history_path}')

payload = json.loads(history_path.read_text(encoding='utf-8'))
df = pd.DataFrame(payload.get('history_clean', []))

if df.empty:
    raise ValueError('Sem dados para plotar. O treino terminou corretamente?')

# 2. Plot: Acurácia Global
plt.figure(figsize=(8, 4))
plt.plot(df['round'], df['accuracy'], marker='o', color='#00b894', linewidth=2)
plt.title('Acuracia Global vs Rodadas de Comunicacao (15dB SNR)')
plt.xlabel('Rodada')
plt.ylabel('Acuracia Global')
plt.grid(alpha=0.3)
plt.xticks(df['round']) # Garante que o eixo X mostre números inteiros (1, 2, 3...)
plt.show()

# 3. Plot: Loss Global
plt.figure(figsize=(8, 4))
plt.plot(df['round'], df['loss'], marker='o', color='#e17055', linewidth=2)
plt.title('Loss Global vs Rodadas de Comunicacao (15dB SNR)')
plt.xlabel('Rodada')
plt.ylabel('Loss Global')
plt.grid(alpha=0.3)
plt.xticks(df['round'])
plt.show()

# 4. Eficiência Semântica (Seus cálculos mantidos iguais)
original_bytes = 3136
latent_bytes_int8 = 36
compression_ratio = original_bytes / latent_bytes_int8
bandwidth_reduction = (1 - latent_bytes_int8 / original_bytes) * 100

plt.figure(figsize=(6, 4))
plt.bar(['Original\n(3136 bytes)', 'Latente Int8\n(36 bytes)'], 
        [original_bytes, latent_bytes_int8], 
        color=['#636e72', '#0984e3'])
plt.title('Proxy de Eficiencia Semantica (Comunicação)')
plt.ylabel('Bytes por amostra')
plt.grid(axis='y', alpha=0.3)
plt.show()

print(f'Compression ratio (MNIST proxy): {compression_ratio:.2f}x')
print(f'Bandwidth reduction (MNIST proxy): {bandwidth_reduction:.1f}%')

AttributeError: module 'matplotlib' has no attribute 'get_data_path'